<a href="https://colab.research.google.com/github/Prathameshshelke145/pcb_defect_detection/blob/main/pcb_defect_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q ultralytics keras-cv opencv-python-headless pillow


In [ ]:
import torch


In [ ]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt

fatal: destination path 'yolov5' already exists and is not an empty directory.
/content/yolov5


In [ ]:
# Cell 3a: install localtunnel (may take a moment)
!npm install -g localtunnel


⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
changed 22 packages in 3s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼

In [ ]:
from google.colab import files

uploaded = files.upload()  # Select multiple files
print(uploaded.keys())     # List of file names


KeyboardInterrupt: 

In [ ]:
import os
import cv2
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import load_model
from tensorflow.keras.utils import load_img, img_to_array
from keras.models import Sequential
from keras.layers import RandomFlip
from keras_cv.layers import (
    RandomSharpness,
    RandomBrightness,
    Resizing
)

from ultralytics import YOLO


In [ ]:
# -------- CLASS NAMES --------

# CNN (mounted PCB, cropped components)
CNN_CLASS_NAMES = [
    'Corrected_BR',
    'Corrected_IC2',
    'Corrected_R7',
    'Corrected_iC1',
    'defected_BR!',
    'miss_align_ic1',
    'miss_aligned_ic2',
    'missing_R7',
    'shifted_R7',
    'white_R7'
]
class_names = CNN_CLASS_NAMES  # alias for compatibility

# YOLO bare PCB (unmounted)
YOLO_BARE_CLASS_NAMES = [
    'missing_hole',
    'mouse_bite',
    'open_circuit',
    'short',
    'spur',
    'spurious_copper'
]

# YOLO mounted PCB (with components)
YOLO_MOUNTED_CLASS_NAMES = [
    'correct',
    'glue',
    'missalign',
    'missing',
    'r_shift',
    'shifted',
    'upsidedown'
]

# -------- MODEL PATHS --------
# (update these to your actual filenames in /content or Drive)
CNN_MODEL_PATH          = "/content/drive/MyDrive/cnn_model_pcb.keras"
YOLO_BARE_MODEL_PATH    = "/content/drive/MyDrive/yolo_bare_pcb2.pt"
YOLO_MOUNTED_MODEL_PATH = "/content/drive/MyDrive/yolo_full_pcb.pt"


In [ ]:
# ------- CNN augmentation / preprocessing pipe (same as your original) -------

pipe = Sequential([
    RandomFlip(mode=np.random.choice(["horizontal", "vertical"])),
    RandomSharpness(factor=(0.0, 0.5), value_range=(0, 255)),
    RandomBrightness(value_range=(0, 255), factor=(-0.2, 0.2)),
    Resizing(256, 256)
])

# ------- Lazy model loaders -------

cnn_model = None
yolo_bare_v5_model = None
yolo_mounted_model = None

def load_cnn_model():
    global cnn_model
    if cnn_model is None:
        print("Loading CNN model from:", CNN_MODEL_PATH)
        cnn_model = load_model(CNN_MODEL_PATH)
    return cnn_model


def load_yolo_bare():
    """
    Load YOLOv5 bare PCB model using torch.hub
    (this is your original code, wrapped in a function).
    """
    global yolo_bare_v5_model
    if yolo_bare_v5_model is None:
        print("Loading YOLOv5 bare PCB model from:", YOLO_BARE_MODEL_PATH)
        yolo_bare_v5_model = torch.hub.load(
            '.',           # local repo with hubconf.py
            'custom',
            path=YOLO_BARE_MODEL_PATH,
            source='local'
        )
    return yolo_bare_v5_model

def load_yolo_mounted():
    global yolo_mounted_model
    if yolo_mounted_model is None:
        print("Loading YOLO MOUNTED model from:", YOLO_MOUNTED_MODEL_PATH)
        yolo_mounted_model = YOLO(YOLO_MOUNTED_MODEL_PATH)
    return yolo_mounted_model


In [ ]:
from google.colab import files
from IPython.display import display, Image as IPyImage

def upload_pcb_image():
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No file uploaded.")
    fname = list(uploaded.keys())[0]
    print("Uploaded:", fname)
    display(IPyImage(filename=fname))
    return os.path.join("/content", fname)

def crop_pcb_for_cnn(image_bgr):
    """
    EXACT same logic as your original:
    - 4 quadrants
    - center strip: height = (h/2)/4, cy = 0.8 * (h/2)
    """
    h, w, _ = image_bgr.shape

    vertical_split = w // 2
    horizontal_split = h // 2

    top_left     = image_bgr[0:horizontal_split,         0:vertical_split]
    top_right    = image_bgr[0:horizontal_split,         vertical_split:w]
    bottom_left  = image_bgr[horizontal_split:h,         0:vertical_split]
    bottom_right = image_bgr[horizontal_split:h,         vertical_split:w]

    center_height = horizontal_split // 4
    cy = int(0.8 * horizontal_split)

    y1 = max(cy - center_height // 2, 0)
    y2 = min(cy + center_height, h)

    center_strip = image_bgr[y1:y2, 0:w]

    return {
        "top_left": top_left,
        "top_right": top_right,
        "bottom_left": bottom_left,
        "bottom_right": bottom_right,
        "center": center_strip,
    }

def save_crops_to_files(crops, prefix="/content/pcb"):
    """
    Save crops in fixed order to disk.
    Returns list of file paths (bottom_right,...,top_right) for compatibility
    with your old input_list order.
    """
    order = ["bottom_right", "bottom_left", "center", "top_left", "top_right"]
    paths = []
    for name in order:
        img = crops[name]
        fname = f"{prefix}_{name}.jpg"
        cv2.imwrite(fname, img)
        print("Saved:", fname)
        paths.append(fname)
    return paths


In [ ]:
def crop_into_parts(image_bgr, center_strip_height_ratio=None):
    """
    For UI display. Uses EXACT original logic.
    center_strip_height_ratio is ignored (kept for API compatibility).
    """
    h, w, _ = image_bgr.shape

    vertical_split = w // 2
    horizontal_split = h // 2

    top_left     = image_bgr[0:horizontal_split,         0:vertical_split]
    top_right    = image_bgr[0:horizontal_split,         vertical_split:w]
    bottom_left  = image_bgr[horizontal_split:h,         0:vertical_split]
    bottom_right = image_bgr[horizontal_split:h,         vertical_split:w]

    center_height = horizontal_split // 4
    cy = int(0.8 * horizontal_split)

    y1 = max(cy - center_height // 2, 0)
    y2 = min(cy + center_height, h)

    center_strip = image_bgr[y1:y2, 0:w]

    return {
        "top_left": top_left,
        "top_right": top_right,
        "bottom_left": bottom_left,
        "bottom_right": bottom_right,
        "center": center_strip,
    }


In [ ]:
import os
import tensorflow as tf

def run_cnn_on_crops_filelist(img_paths):
    """
    img_paths: list of file paths like:
      /content/pcb_debug_top_left.jpg
      /content/pcb_debug_bottom_right.jpg
      /content/pcb_debug_center.jpg
    Returns:
      {
        "top_left":  {"image_path": ..., "class": ..., "confidence": ...},
        "top_right": {...},
        "bottom_left": {...},
        "bottom_right": {...},
        "center": {...}
      }
    (only keys actually found in filenames will appear)
    """
    model = load_cnn_model()
    results = {}

    # standard region names we care about
    REGION_KEYS = ["top_left", "top_right", "bottom_left", "bottom_right", "center"]

    for img_path in img_paths:
        base = os.path.basename(img_path).lower()

        # infer region from filename
        region = None
        for key in REGION_KEYS:
            if key in base:
                region = key
                break

        # if nothing matched, skip this file
        if region is None:
            print(f"[CNN] Skipping file (no region keyword): {base}")
            continue

        # --- preprocess & predict (your original logic style) ---
        img = tf.keras.utils.load_img(img_path)
        img = tf.keras.utils.img_to_array(img)
        img = tf.expand_dims(img, 0)

        # pipe: your augmentation + resizing Sequential
        img_processed = pipe(img, training=False)

        pred = model.predict(img_processed, verbose=0)
        prob = tf.nn.softmax(pred[0])
        class_id = int(tf.argmax(prob).numpy())
        confidence = float(tf.reduce_max(prob).numpy())
        label = CNN_CLASS_NAMES[class_id]

        # store using the full region name (no collapsing left/right)
        results[region] = {
            "image_path": img_path,
            "class": label,
            "confidence": confidence
        }

        print("------------------------------------------------------------------------------")
        print(f"Region      : {region}")
        print(f"Image path  : {img_path}")
        print(f"CNN class   : {label}")
        print(f"confidence  : {confidence}")
        print("------------------------------------------------------------------------------")

    return results


In [ ]:
def run_yolo_detection(image_path, model_type="mounted", conf=0.25, iou=0.45):
    """
    model_type: 'bare'    -> YOLOv5 torch.hub model (best.pt)
                'mounted' -> Ultralytics YOLO (YOLOv8 style)

    Returns: (detections_list, annotated_image_path)

    Each detection:
      {
        "class_id": int,
        "label": str,
        "confidence": float,
        "bbox_xyxy": [x1, y1, x2, y2]
      }
    """
    detections = []

    if model_type == "bare":
        # ---------- YOLOv5 (torch.hub) path ----------
        model = load_yolo_bare()

        # optional: set thresholds on the model itself
        try:
            model.conf = conf
            model.iou = iou
        except Exception:
            pass  # older versions may not support this, it's fine

        # run inference
        results = model(image_path)

        # --- build detections from pandas (this part is same as your original style) ---
        df = results.pandas().xyxy[0]  # xmin, ymin, xmax, ymax, conf, class, name

        for _, row in df.iterrows():
            x1, y1, x2, y2 = row["xmin"], row["ymin"], row["xmax"], row["ymax"]
            score = float(row["confidence"])
            cls_id = int(row["class"])
            label = str(row["name"])

            detections.append({
                "class_id": cls_id,
                "label": label,
                "confidence": round(score, 3),
                "bbox_xyxy": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]
            })

        # --- get annotated image without using results.imgs ---
        try:
            rendered = results.render()  # in most YOLOv5 versions this returns list of RGB images
            if isinstance(rendered, list) and len(rendered) > 0:
                annotated_rgb = rendered[0]
            else:
                annotated_rgb = rendered
            annotated_bgr = annotated_rgb[:, :, ::-1]  # RGB -> BGR
        except Exception:
            # fallback: if something weird, just use the original image
            annotated_bgr = cv2.imread(image_path)

        base, ext = os.path.splitext(image_path)
        out_path = f"{base}_{model_type}_yolo.jpg"
        cv2.imwrite(out_path, annotated_bgr)

        print(f"✅ YOLOv5 [bare] done. Annotated: {out_path}")
        for d in detections:
            print(d)

        return detections, out_path

    # ---------- Ultralytics YOLOv8 path for mounted PCB ----------
    else:
        model = load_yolo_mounted()
        class_list = YOLO_MOUNTED_CLASS_NAMES

        results = model.predict(
            source=image_path,
            conf=conf,
            iou=iou,
            verbose=False,
        )
        result = results[0]

        annotated = result.plot()
        base, ext = os.path.splitext(image_path)
        out_path = f"{base}_{model_type}_yolo.jpg"
        cv2.imwrite(out_path, annotated)

        boxes = result.boxes
        if boxes is not None:
            for box in boxes:
                cls_id = int(box.cls[0].item())
                score = float(box.conf[0].item())
                x1, y1, x2, y2 = box.xyxy[0].tolist()

                label = class_list[cls_id] if 0 <= cls_id < len(class_list) else f"class_{cls_id}"

                detections.append({
                    "class_id": cls_id,
                    "label": label,
                    "confidence": round(score, 3),
                    "bbox_xyxy": [round(x1, 1), round(y1, 1), round(x2, 1), round(y2, 1)]
                })

        print(f"✅ YOLOv8 [mounted] done. Annotated: {out_path}")
        for d in detections:
            print(d)

        return detections, out_path


In [ ]:
# YOLO mounted: everything except "correct" is defect
DEFECT_LABELS_YOLO_MOUNTED = [lbl for lbl in YOLO_MOUNTED_CLASS_NAMES if lbl != "correct"]

# YOLO bare: all classes are defects
DEFECT_LABELS_YOLO_BARE = YOLO_BARE_CLASS_NAMES.copy()

# CNN: non-defect classes
NON_DEFECT_CNN_LABELS = [
    "Corrected_BR",
    "Corrected_IC2",
    "Corrected_R7",
    "Corrected_iC1",
]
DEFECT_LABELS_CNN = [lbl for lbl in CNN_CLASS_NAMES if lbl not in NON_DEFECT_CNN_LABELS]

print("Mounted YOLO defect labels:", DEFECT_LABELS_YOLO_MOUNTED)
print("Bare YOLO defect labels   :", DEFECT_LABELS_YOLO_BARE)
print("CNN defect labels         :", DEFECT_LABELS_CNN)


Mounted YOLO defect labels: ['glue', 'missalign', 'missing', 'r_shift', 'shifted', 'upsidedown']
Bare YOLO defect labels   : ['missing_hole', 'mouse_bite', 'open_circuit', 'short', 'spur', 'spurious_copper']
CNN defect labels         : ['defected_BR!', 'miss_align_ic1', 'miss_aligned_ic2', 'missing_R7', 'shifted_R7', 'white_R7']


In [ ]:
def yolo_has_defect(detections, defect_labels):
    for det in detections:
        if det["label"] in defect_labels:
            return True
    return False

def cnn_has_defect(cnn_results, defect_labels):
    for region, info in cnn_results.items():
        if info["class"] in defect_labels:
            return True
    return False


In [ ]:
def analyze_pcb_bare(image_path, conf=0.25, iou=0.45):
    """
    Bare PCB (no components): YOLOv5 bare model detects *only* defects.

    => If there is at least ONE detection, board is DEFECTIVE.
       If no detections, board is OK.
    """
    report = {
        "original_image": image_path,
        "board_type": "bare",
        "status": None,
        "reason": "",
        "yolo_initial": {},
        "cnn_stage": {},      # unused for bare
        "yolo_recheck": {},   # unused for bare
    }

    print("\n=== BARE PCB: YOLOv5 bare detection ===")
    dets, annot = run_yolo_detection(
        image_path,
        model_type="bare",
        conf=conf,
        iou=iou
    )

    report["yolo_initial"] = {
        "detections": dets,
        "annotated_image": annot,
        "conf": conf,
    }

    if len(dets) > 0:
        # any detection means some defect (spur, missing_hole, etc.)
        report["status"] = "DEFECTIVE"
        report["reason"] = (
            "YOLOv5 bare PCB model detected one or more defect classes: "
            + ", ".join(sorted({d['label'] for d in dets}))
        )
        print("\n❌ BARE PCB DEFECTIVE: detections found.")
    else:
        report["status"] = "OK"
        report["reason"] = "YOLOv5 bare PCB model found no defects."
        print("\n✅ BARE PCB OK: no detections.")

    return report


In [ ]:
def analyze_pcb_mounted_smart(
    image_path,
    initial_conf=0.25,
    recheck_conf=0.15,
    iou=0.45,
):
    report = {
        "original_image": image_path,
        "board_type": "mounted",
        "status": None,
        "reason": "",
        "yolo_initial": {},
        "cnn_stage": {},
        "yolo_recheck": {},
    }

    # STEP 1: YOLO mounted initial
    print("\n=== STEP 1: YOLO initial (mounted) ===")
    dets1, annot1 = run_yolo_detection(
        image_path,
        model_type="mounted",
        conf=initial_conf,
        iou=iou
    )
    report["yolo_initial"] = {
        "detections": dets1,
        "annotated_image": annot1,
        "conf": initial_conf,
    }

    if yolo_has_defect(dets1, DEFECT_LABELS_YOLO_MOUNTED):
        report["status"] = "DEFECTIVE"
        report["reason"] = "YOLO mounted model detected defect on first pass."
        print("\n❌ DEFECTIVE: YOLO initial found defect.")
        return report

    print("\n✅ YOLO initial: no defect. Proceeding to CNN...")

    # STEP 2: CNN on crops
    print("\n=== STEP 2: CNN on cropped regions ===")
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Could not read image at " + image_path)

    crops = crop_pcb_for_cnn(img)
    crop_files = save_crops_to_files(crops, prefix="/content/pcb")
    cnn_results = run_cnn_on_crops_filelist(crop_files)

    # 🔹 Ensure cnn_stage has all 5 standard keys if available
    ordered_keys = ["top_left", "top_right", "bottom_left", "bottom_right", "center"]
    cnn_stage = {}
    for k in ordered_keys:
        if k in cnn_results:
            cnn_stage[k] = cnn_results[k]
    # also keep any legacy keys like "left"/"right" if your runner uses them
    for k in ["left", "right"]:
        if k in cnn_results and k not in cnn_stage:
            cnn_stage[k] = cnn_results[k]

    report["cnn_stage"] = cnn_stage

    if not cnn_has_defect(cnn_results, DEFECT_LABELS_CNN):
        report["status"] = "OK"
        report["reason"] = "Neither YOLO initial nor CNN found defects."
        print("\n✅ PCB OK: CNN also found no defect.")
        return report

    print("\n⚠️ CNN reports possible defect. Rechecking with YOLO (more sensitive)...")

    # STEP 3: YOLO recheck (lower conf)
    print("\n=== STEP 3: YOLO recheck (mounted, lower conf) ===")
    dets2, annot2 = run_yolo_detection(
        image_path,
        model_type="mounted",
        conf=recheck_conf,
        iou=iou
    )
    report["yolo_recheck"] = {
        "detections": dets2,
        "annotated_image": annot2,
        "conf": recheck_conf,
    }

    if yolo_has_defect(dets2, DEFECT_LABELS_YOLO_MOUNTED):
        report["status"] = "DEFECTIVE"
        report["reason"] = "CNN suspected defect and YOLO recheck confirmed it."
        print("\n❌ DEFECTIVE: YOLO recheck confirmed defect after CNN.")
        return report

    report["status"] = "OK_WITH_WARNING"
    report["reason"] = (
        "CNN predicted a defect class, but both YOLO passes showed no defect. "
        "Treating PCB as OK with a warning."
    )
    print("\n✅ YOLO recheck clean: treating CNN as possible false alarm.")
    return report


In [ ]:
def analyze_pcb(image_path, board_type="mounted", **kwargs):
    board_type = board_type.lower()
    if board_type == "bare":
        return analyze_pcb_bare(
            image_path,
            conf=kwargs.get("conf", 0.25),
            iou=kwargs.get("iou", 0.45)
        )
    else:
        return analyze_pcb_mounted_smart(
            image_path,
            initial_conf=kwargs.get("initial_conf", 0.25),
            recheck_conf=kwargs.get("recheck_conf", 0.15),
            iou=kwargs.get("iou", 0.45)
        )


In [ ]:
import json
import cv2

def debug_cnn_on_pcb(image_path):
    """
    Runs ONLY the CNN part on all crops and prints per-region outputs.
    This ignores YOLO completely, just to verify which regions CNN returns.
    """
    print("\n=== DEBUG: CNN on cropped regions only ===")

    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read image at {image_path}")

    # your existing cropping+save logic
    crops = crop_pcb_for_cnn(img)                      # returns dict of crops
    crop_files = save_crops_to_files(crops, prefix="/content/pcb_debug")
    cnn_results = run_cnn_on_crops_filelist(crop_files)

    print("\nRaw cnn_results dict:")
    print(json.dumps(cnn_results, indent=2))

    # Check all standard regions explicitly
    print("\nPer-region summary:")
    for key in ["top_left", "top_right", "bottom_left", "bottom_right", "center", "left", "right"]:
        info = cnn_results.get(key)
        if info is None:
            print(f"  {key}:  NO OUTPUT")
        else:
            print(f"  {key}:  class={info.get('class')}  conf={info.get('confidence'):.3f}  img={info.get('image_path')}")

    return cnn_results


In [ ]:
# Upload a test PCB image (mounted)
IMAGE_PATH = "/content/ok_full_aug_18.jpg"

cnn_debug = debug_cnn_on_pcb(IMAGE_PATH)



=== DEBUG: CNN on cropped regions only ===
Saved: /content/pcb_debug_bottom_right.jpg
Saved: /content/pcb_debug_bottom_left.jpg
Saved: /content/pcb_debug_center.jpg
Saved: /content/pcb_debug_top_left.jpg
Saved: /content/pcb_debug_top_right.jpg
------------------------------------------------------------------------------
Region      : bottom_right
Image path  : /content/pcb_debug_bottom_right.jpg
CNN class   : Corrected_BR
confidence  : 0.23195835947990417
------------------------------------------------------------------------------
------------------------------------------------------------------------------
Region      : bottom_left
Image path  : /content/pcb_debug_bottom_left.jpg
CNN class   : Corrected_iC1
confidence  : 0.23196931183338165
------------------------------------------------------------------------------
------------------------------------------------------------------------------
Region      : center
Image path  : /content/pcb_debug_center.jpg
CNN class   : Correc

In [ ]:
image_path = "/content/08.JPG"  # your bare PCB test image

report_bare = analyze_pcb(
    image_path,
    board_type="bare",
    conf=0.25,
    iou=0.45
)

print("Status:", report_bare["status"])
print("Reason:", report_bare["reason"])
print("Detections:", report_bare["yolo_initial"]["detections"])



=== BARE PCB: YOLOv5 bare detection ===


/content/yolov5/models/common.py:898: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):


✅ YOLOv5 [bare] done. Annotated: /content/08_bare_yolo.jpg
{'class_id': 0, 'label': 'missing_hole', 'confidence': 0.843, 'bbox_xyxy': [1571.4, 829.9, 1619.2, 875.8]}
{'class_id': 0, 'label': 'missing_hole', 'confidence': 0.728, 'bbox_xyxy': [2287.2, 1164.8, 2334.5, 1208.0]}
{'class_id': 0, 'label': 'missing_hole', 'confidence': 0.589, 'bbox_xyxy': [1777.5, 683.8, 1825.4, 728.3]}
{'class_id': 0, 'label': 'missing_hole', 'confidence': 0.386, 'bbox_xyxy': [1721.4, 1202.2, 1762.5, 1248.3]}
{'class_id': 0, 'label': 'missing_hole', 'confidence': 0.331, 'bbox_xyxy': [2037.4, 1951.8, 2075.2, 1988.2]}
{'class_id': 0, 'label': 'missing_hole', 'confidence': 0.301, 'bbox_xyxy': [2226.6, 778.2, 2274.5, 821.2]}

❌ BARE PCB DEFECTIVE: detections found.
Status: DEFECTIVE
Reason: YOLOv5 bare PCB model detected one or more defect classes: missing_hole
Detections: [{'class_id': 0, 'label': 'missing_hole', 'confidence': 0.843, 'bbox_xyxy': [1571.4, 829.9, 1619.2, 875.8]}, {'class_id': 0, 'label': 'missing

In [ ]:
# ------------------------- Cell 9: Flask UI -------------------------
!pip install -q flask==2.2.5 pyngrok==5.2.1 opencv-python-headless pillow

import io, base64, json, tempfile, threading, time, re, random, shutil, subprocess, sys
from flask import Flask, request, render_template_string
from PIL import Image

# Put your full HTML string here (with the added board_type <select>)
HTML = """ <!DOCTYPE html>
<html lang="en">

<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <meta name="description"
        content="Professional PCB Quality Inspection System - Advanced AI-powered circuit board analysis">
    <meta name="keywords" content="PCB, inspection, quality control, AI, YOLO, CNN">
    <title>PCB Quality Inspector | Professional Circuit Board Analysis</title>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap" rel="stylesheet">
    <link href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css" rel="stylesheet">
    <style>
        :root {
            --primary-color: #0969da;
            --primary-dark: #0550ae;
            --secondary-color: #656d76;
            --success-color: #1a7f37;
            --error-color: #cf222e;
            --warning-color: #bf8700;
            --background-color: #f6f8fa;
            --surface-color: #ffffff;
            --surface-secondary: #f6f8fa;
            --text-primary: #1f2328;
            --text-secondary: #656d76;
            --text-muted: #8c959f;
            --border-color: #d1d9e0;
            --border-muted: #d8dee4;
            --shadow-sm: 0 1px 2px rgba(31, 35, 40, 0.04);
            --shadow-md: 0 3px 6px rgba(140, 149, 159, 0.15);
            --shadow-lg: 0 8px 24px rgba(140, 149, 159, 0.2);
            --radius-sm: 6px;
            --radius-md: 8px;
            --radius-lg: 12px;
        }

        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', 'Noto Sans', Helvetica, Arial, sans-serif;
            background: var(--background-color);
            color: var(--text-primary);
            line-height: 1.5;
            font-size: 14px;
            min-height: 100vh;
        }

        .container {
            max-width: 1200px;
            margin: 0 auto;
            padding: 24px;
        }

        .header {
            margin-bottom: 32px;
            padding-bottom: 16px;
            border-bottom: 1px solid var(--border-color);
        }

        .header h1 {
            font-size: 32px;
            font-weight: 600;
            color: var(--text-primary);
            margin-bottom: 4px;
            display: flex;
            align-items: center;
            gap: 12px;
        }

        .header h1 i {
            color: var(--primary-color);
            font-size: 28px;
        }

        .header .subtitle {
            font-size: 16px;
            color: var(--text-secondary);
            font-weight: 400;
        }

        .form-section {
            background: var(--surface-color);
            padding: 24px;
            border-radius: var(--radius-md);
            border: 1px solid var(--border-color);
            margin-bottom: 24px;
        }

        .form-title {
            font-size: 20px;
            font-weight: 600;
            color: var(--text-primary);
            margin-bottom: 20px;
            display: flex;
            align-items: center;
            gap: 8px;
        }

        .form-title i {
            color: var(--text-secondary);
            font-size: 16px;
        }

        .form-grid {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 16px;
            margin-bottom: 20px;
        }

        .form-group {
            display: flex;
            flex-direction: column;
        }

        .form-group label {
            font-weight: 500;
            color: var(--text-primary);
            margin-bottom: 6px;
            font-size: 14px;
            display: flex;
            align-items: center;
            gap: 6px;
        }

        .form-group label i {
            color: var(--text-secondary);
            font-size: 12px;
        }

        .form-group input {
            padding: 8px 12px;
            border: 1px solid var(--border-color);
            border-radius: var(--radius-sm);
            font-size: 14px;
            transition: all 0.15s ease;
            background: var(--surface-color);
        }

        .form-group input:focus {
            outline: none;
            border-color: var(--primary-color);
            box-shadow: 0 0 0 3px rgba(9, 105, 218, 0.12);
        }

        /* 🔹 NEW: style for board type <select> */
        .form-group select {
            padding: 8px 12px;
            border: 1px solid var(--border-color);
            border-radius: var(--radius-sm);
            font-size: 14px;
            transition: all 0.15s ease;
            background: var(--surface-color);
        }

        .form-group select:focus {
            outline: none;
            border-color: var(--primary-color);
            box-shadow: 0 0 0 3px rgba(9, 105, 218, 0.12);
        }

        .file-input-wrapper {
            position: relative;
            display: inline-block;
            width: 100%;
        }

        .file-input {
            position: absolute;
            opacity: 0;
            width: 100%;
            height: 100%;
            cursor: pointer;
        }

        .file-input-display {
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 32px 16px;
            border: 2px dashed var(--border-muted);
            border-radius: var(--radius-md);
            background: var(--surface-secondary);
            transition: all 0.15s ease;
            cursor: pointer;
            text-align: center;
        }

        .file-input-display:hover {
            border-color: var(--primary-color);
            background: var(--surface-color);
        }

        .btn {
            background: var(--primary-color);
            color: white;
            padding: 10px 20px;
            border: none;
            border-radius: var(--radius-sm);
            font-size: 14px;
            font-weight: 500;
            cursor: pointer;
            transition: all 0.15s ease;
            display: inline-flex;
            align-items: center;
            gap: 8px;
        }

        .btn:hover {
            background: var(--primary-dark);
        }

        .btn:active {
            transform: scale(0.98);
        }

        .results-section {
            margin-top: 24px;
        }

        .wrap {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 24px;
            margin-top: 16px;
        }

        .col {
            background: var(--surface-color);
            padding: 20px;
            border-radius: var(--radius-md);
            border: 1px solid var(--border-color);
        }

        .col h3 {
            font-size: 18px;
            font-weight: 600;
            color: var(--text-primary);
            margin-bottom: 16px;
            padding-bottom: 8px;
            border-bottom: 1px solid var(--border-color);
            display: flex;
            align-items: center;
            gap: 8px;
        }

        .col h3 i {
            color: var(--text-secondary);
            font-size: 16px;
        }

        .parts {
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(120px, 1fr));
            gap: 12px;
            margin-top: 12px;
        }

        .part {
            text-align: center;
            padding: 12px;
            background: var(--surface-secondary);
            border-radius: var(--radius-sm);
            border: 1px solid var(--border-muted);
        }

        .part img {
            width: 100%;
            height: auto;
            border-radius: 4px;
            margin-bottom: 8px;
        }

        img {
            max-width: 100%;
            height: auto;
            border-radius: var(--radius-sm);
            border: 1px solid var(--border-muted);
        }

        pre {
            background: #0d1117;
            color: #7c3aed;
            padding: 16px;
            border-radius: var(--radius-sm);
            overflow: auto;
            max-height: 300px;
            font-family: 'SF Mono', Monaco, 'Cascadia Code', 'Roboto Mono', Consolas, monospace;
            font-size: 12px;
            line-height: 1.45;
            border: 1px solid #21262d;
        }

        .verdict-good {
            background: #dafbe1;
            color: var(--success-color);
            padding: 12px 16px;
            border-radius: var(--radius-sm);
            font-weight: 500;
            border: 1px solid #aceeba;
            display: flex;
            align-items: center;
            gap: 8px;
            font-size: 14px;
        }

        .verdict-bad {
            background: #ffebe9;
            color: var(--error-color);
            padding: 12px 16px;
            border-radius: var(--radius-sm);
            font-weight: 500;
            border: 1px solid #ffbdad;
            display: flex;
            align-items: center;
            gap: 8px;
            font-size: 14px;
        }

        .small {
            font-size: 12px;
            color: var(--text-secondary);
            font-weight: 400;
        }

        .footer {
            text-align: center;
            margin-top: 48px;
            padding: 24px;
            border-top: 1px solid var(--border-color);
        }

        .divider {
            height: 1px;
            background: var(--border-color);
            margin: 32px 0;
        }

        @media (max-width: 768px) {
            .container {
                padding: 16px;
            }

            .wrap {
                grid-template-columns: 1fr;
                gap: 16px;
            }

            .form-grid {
                grid-template-columns: 1fr;
            }

            .header h1 {
                font-size: 24px;
            }

            .col {
                padding: 16px;
            }
        }

        .loading {
            display: none;
            text-align: center;
            padding: 2rem;
        }

        .spinner {
            border: 3px solid var(--border-color);
            border-top: 3px solid var(--primary-color);
            border-radius: 50%;
            width: 40px;
            height: 40px;
            animation: spin 1s linear infinite;
            margin: 0 auto 1rem;
        }

        @keyframes spin {
            0% {
                transform: rotate(0deg);
            }

            100% {
                transform: rotate(360deg);
            }
        }
    </style>
</head>

<body>
    <div class="container">
        <header class="header">
            <h1>
                <i class="fas fa-microchip"></i>
                PCB Quality Inspector
            </h1>
            <p class="subtitle">Professional Circuit Board Analysis & Quality Control System</p>
        </header>

        <section class="form-section">
            <h2 class="form-title">
                <i class="fas fa-upload"></i>
                Upload & Configure Analysis
            </h2>

            <form method="post" action="/inspect" enctype="multipart/form-data" id="inspectionForm">
                <div class="form-grid">
                    <div class="form-group">
                        <label for="image">
                            <i class="fas fa-image"></i>
                            Upload PCB Image
                        </label>
                        <div class="file-input-wrapper">
                            <input type="file" name="image" id="image" class="file-input" accept="image/*" required />
                            <div class="file-input-display">
                                <div>
                                    <i class="fas fa-cloud-upload-alt"
                                        style="font-size: 2rem; color: var(--primary-color); margin-bottom: 0.5rem;"></i>
                                    <p><strong>Click to upload</strong> or drag and drop</p>
                                    <p class="small">Supports JPG, PNG, BMP formats</p>
                                </div>
                            </div>
                        </div>
                    </div>
                </div>

                <div class="form-grid">
                    <!-- 🔹 NEW: Board type selector -->
                    <div class="form-group">
                        <label for="board_type">
                            <i class="fas fa-microchip"></i>
                            Board Type
                        </label>
                        <select name="board_type" id="board_type">
                            <option value="mounted" selected>Mounted (YOLO + CNN)</option>
                            <option value="bare">Bare PCB (YOLO Bare only)</option>
                        </select>
                    </div>

                    <div class="form-group">
                        <label for="yolo_init_conf">
                            <i class="fas fa-search"></i>
                            YOLO Initial Confidence
                        </label>
                        <input name="yolo_init_conf" id="yolo_init_conf" type="number" min="0.05" max="0.9" step="0.01"
                            value="0.25" />
                    </div>

                    <div class="form-group">
                        <label for="yolo_recheck_conf">
                            <i class="fas fa-redo"></i>
                            YOLO Recheck Confidence
                        </label>
                        <input name="yolo_recheck_conf" id="yolo_recheck_conf" type="number" min="0.05" max="0.9"
                            step="0.01" value="0.15" />
                    </div>

                    <div class="form-group">
                        <label for="center_strip_ratio">
                            <i class="fas fa-expand-arrows-alt"></i>
                            Center Strip Ratio
                        </label>
                        <input name="center_strip_ratio" id="center_strip_ratio" type="number" min="0.1" max="0.8"
                            step="0.05" value="0.25" />
                    </div>
                </div>

                <div style="text-align: center; margin-top: 24px;">
                    <button type="submit" class="btn">
                        <i class="fas fa-play"></i>
                        Start Quality Inspection
                    </button>
                </div>
            </form>

            <div class="loading" id="loadingIndicator">
                <div class="spinner"></div>
                <p>Processing PCB analysis... Please wait.</p>
            </div>
        </section>

        {% if results %}
        <div class="divider"></div>

        <section class="results-section">
            <div class="wrap">
                <div class="col">
                    <h3>
                        <i class="fas fa-images"></i>
                        Analysis Results
                    </h3>

                    <div style="margin-bottom: 20px;">
                        <h4 style="color: var(--text-secondary); margin-bottom: 8px; font-size: 14px; font-weight: 500;">Original Image</h4>
                        <img src="data:image/jpeg;base64,{{ results.original_b64 }}" alt="Original PCB Image" />
                    </div>

                    <div style="margin-bottom: 20px;">
                        <h4 style="color: var(--text-secondary); margin-bottom: 8px; font-size: 14px; font-weight: 500;">YOLO Initial Detection</h4>
                        {% if results.yolo_init_b64 %}
                        <img src="data:image/png;base64,{{ results.yolo_init_b64 }}" alt="YOLO Initial Detection" />
                        {% else %}
                        <div class="small"
                            style="padding: 16px; text-align: center; background: var(--surface-secondary); border-radius: var(--radius-sm); border: 1px solid var(--border-muted);">
                            <i class="fas fa-info-circle"></i> No initial detections found
                        </div>
                        {% endif %}
                    </div>

                    <div style="margin-bottom: 20px;">
                        <h4 style="color: var(--text-secondary); margin-bottom: 8px; font-size: 14px; font-weight: 500;">YOLO Recheck Detection</h4>
                        {% if results.yolo_re_b64 %}
                        <img src="data:image/png;base64,{{ results.yolo_re_b64 }}" alt="YOLO Recheck Detection" />
                        {% else %}
                        <div class="small"
                            style="padding: 16px; text-align: center; background: var(--surface-secondary); border-radius: var(--radius-sm); border: 1px solid var(--border-muted);">
                            <i class="fas fa-info-circle"></i> No recheck detections found
                        </div>
                        {% endif %}
                    </div>

                    <div>
                        <h4 style="color: var(--text-secondary); margin-bottom: 8px; font-size: 14px; font-weight: 500;">Quality Assessment</h4>
                        {% if results.final_verdict == "GOOD" %}
                        <div class="verdict-good">
                            <i class="fas fa-check-circle"></i>
                            PCB Quality: PASSED
                        </div>
                        {% elif results.final_verdict == "DEFECTIVE" %}
                        <div class="verdict-bad">
                            <i class="fas fa-times-circle"></i>
                            PCB Quality: FAILED
                        </div>
                        {% else %}
                        <div class="small"
                            style="padding: 12px; background: var(--surface-secondary); border-radius: var(--radius-sm); border: 1px solid var(--border-color);">
                            <i class="fas fa-question-circle"></i>
                            Assessment Result: {{ results.final_verdict }}
                        </div>
                        {% endif %}
                    </div>
                </div>

                <div class="col">
                    <h3>
                        <i class="fas fa-cogs"></i>
                        Component Analysis
                    </h3>

                    <div class="parts">
                        {% for p in results.parts %}
                        <div class="part">
                            <img src="data:image/jpeg;base64,{{ p.img_b64 }}" alt="Component {{ p.name }}" />
                            <div style="margin-top: 6px;">
                                <div style="font-weight: 500; color: var(--text-primary); font-size: 12px;">{{ p.name }}</div>
                                <div class="small">{{ p.label }}</div>
                                <div class="small">{{ '%.1f'|format(p.score * 100) }}%</div>
                            </div>
                        </div>
                        {% endfor %}
                    </div>

                    <div style="margin-top: 20px;">
                        <h4 style="color: var(--text-secondary); margin-bottom: 8px; font-size: 14px; font-weight: 500; display: flex; align-items: center; gap: 6px;">
                            <i class="fas fa-code"></i>
                            Technical Details
                        </h4>
                        <pre>{{ results.raw_json }}</pre>
                    </div>
                </div>
            </div>
        </section>
        {% endif %}

        <footer class="footer">
            <p class="small">
                <i class="fas fa-flask"></i>
                Professional PCB Quality Control System | Powered by Advanced AI & Machine Learning
            </p>
            <p class="small" style="margin-top: 0.5rem; opacity: 0.7;">
                This application integrates YOLO object detection and CNN classification for comprehensive circuit board
                analysis
            </p>
        </footer>
    </div>

    <script>
        // Enhanced form interaction
        document.getElementById('inspectionForm').addEventListener('submit', function () {
            document.getElementById('loadingIndicator').style.display = 'block';
        });

        // File input enhancement
        const fileInput = document.getElementById('image');
        const fileDisplay = document.querySelector('.file-input-display');

        fileInput.addEventListener('change', function (e) {
            if (e.target.files.length > 0) {
                const fileName = e.target.files[0].name;
                fileDisplay.innerHTML = `
          <div>
            <i class="fas fa-check-circle" style="font-size: 2rem; color: var(--success-color); margin-bottom: 0.5rem;"></i>
            <p><strong>Selected:</strong> ${fileName}</p>
            <p class="small">Click to change file</p>
          </div>
        `;
            }
        });
    </script>
</body>

</html>
 """

def bgr_to_jpeg_b64(img_bgr, fmt="jpeg", quality=90):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(rgb)
    buff = io.BytesIO()
    pil.save(buff, format=fmt.upper(), quality=quality)
    return base64.b64encode(buff.getvalue()).decode("ascii")

def draw_yolo_boxes(img_bgr, detections, board_type=None):
    """
    detections: list of dicts from our pipeline, e.g.:

      Mounted (YOLOv8):
        {
          "class_id": int,
          "label": str,
          "confidence": float,
          "bbox_xyxy": [x1,y1,x2,y2]
        }

      Bare (YOLOv5):
        same structure, coming from run_yolo_detection(..., model_type="bare")

    board_type: "mounted" or "bare" (optional, only used to color logic)
    """
    img = img_bgr.copy()

    for d in detections:
        try:
            # 1) Get box coords (support both bbox_xyxy and xyxy or raw columns)
            if "bbox_xyxy" in d:
                x1, y1, x2, y2 = d["bbox_xyxy"]
            elif "xyxy" in d:
                x1, y1, x2, y2 = d["xyxy"]
            elif all(k in d for k in ["xmin", "ymin", "xmax", "ymax"]):
                x1, y1, x2, y2 = d["xmin"], d["ymin"], d["xmax"], d["ymax"]
            else:
                continue  # unknown format

            x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

            # 2) Get class label + confidence (support multiple key names)
            label = d.get("label") or d.get("name") or str(d.get("class_id", d.get("class", "?")))
            conf = float(d.get("confidence", d.get("conf", 0.0)))

            text = f"{label}:{conf:.2f}"

            # 3) Color logic:
            #    - Mounted: 'correct' = green, others = red
            #    - Bare: everything is defect -> red boxes
            if board_type == "mounted":
                if label == "correct":
                    color = (0, 255, 0)  # green
                else:
                    color = (0, 0, 255)  # red
            else:
                # bare PCB: all detections are defects (missing_hole, spur, etc.)
                color = (0, 0, 255)

            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, text, (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        except Exception:
            continue  # ignore bad detection entries

    return img


app = Flask(__name__)


@app.route("/shutdown", methods=["GET"])
def shutdown():
    func = request.environ.get("werkzeug.server.shutdown")
    if func is None:
        return "Not running with the Werkzeug Server", 500
    func()
    return "Server shutting down..."


# helper: map UI regions -> cnn_stage keys
def get_cnn_info_for_region(cnn_stage, ui_region_key):
    """
    Map UI region names (top_left, top_right, etc.)
    to whatever keys the backend actually used (left, right, center).
    """
    region_key_map = {
        "top_left": ["top_left", "left"],
        "top_right": ["top_right", "right"],
        "bottom_left": ["bottom_left"],      # no CNN in your example
        "bottom_right": ["bottom_right"],    # no CNN in your example
        "center": ["center"],
    }

    candidates = region_key_map.get(ui_region_key, [ui_region_key])
    for k in candidates:
        if k in cnn_stage:
            return cnn_stage[k]
    return None


@app.route("/", methods=["GET"])
def index():
    return render_template_string(HTML)


@app.route("/inspect", methods=["POST"])
def inspect():
    f = request.files.get("image")
    if f is None:
        return "No file", 400

    tmpdir = tempfile.mkdtemp()
    img_path = os.path.join(tmpdir, f.filename)
    f.save(img_path)

    try:
        yi = float(request.form.get("yolo_init_conf", 0.25))
        yr = float(request.form.get("yolo_recheck_conf", 0.15))
        cs = float(request.form.get("center_strip_ratio", 0.25))
    except Exception:
        yi, yr, cs = 0.25, 0.15, 0.25

    board_type = request.form.get("board_type", "mounted").lower()

    # run pipeline
    try:
        if board_type == "bare":
            res = analyze_pcb(img_path, board_type="bare", conf=yi, iou=0.45)
        else:
            res = analyze_pcb(
                img_path,
                board_type="mounted",
                initial_conf=yi,
                recheck_conf=yr,
                iou=0.45
            )
    except Exception as e:
        return f"<h3>Pipeline exception:</h3><pre>{str(e)}</pre>", 500

    orig_bgr = cv2.imread(img_path)
    original_b64 = bgr_to_jpeg_b64(orig_bgr, fmt="jpeg")

    yolo_init = res.get("yolo_initial", {}).get("detections", [])
    yolo_re = res.get("yolo_recheck", {}).get("detections", [])

    y_init_b64 = None
    if yolo_init:
        vis = draw_yolo_boxes(orig_bgr, yolo_init, board_type=board_type)
        y_init_b64 = bgr_to_jpeg_b64(vis, fmt="png")

    y_re_b64 = None
    if yolo_re:
        vis2 = draw_yolo_boxes(orig_bgr, yolo_re, board_type=board_type)
        y_re_b64 = bgr_to_jpeg_b64(vis2, fmt="png")

    parts_payload = []

    if board_type == "mounted":
        try:
            parts_dict = crop_into_parts(orig_bgr, center_strip_height_ratio=cs)
        except Exception as e:
            return f"<h3>crop_into_parts errored: {e}</h3>", 500

        cnn_stage = res.get("cnn_stage", {})
        region_order = ["top_left", "top_right", "bottom_left", "bottom_right", "center"]
        name_map = {
            "top_left": "Top-Left",
            "top_right": "Top-Right",
            "bottom_left": "Bottom-Left",
            "bottom_right": "Bottom-Right",
            "center": "Center",
        }

        for key in region_order:
            if key not in parts_dict:
                continue
            pimg = parts_dict[key]
            b64 = bgr_to_jpeg_b64(pimg, fmt="jpeg")

            info = get_cnn_info_for_region(cnn_stage, key)
            if info is None:
                label = "CNN not run"
                score = 0.0
            else:
                label = info.get("class", "N/A")
                score = float(info.get("confidence", 0.0))

            parts_payload.append({
                "name": name_map.get(key, key),
                "img_b64": b64,
                "label": label,
                "score": score
            })

    status = res.get("status", "UNKNOWN")
    if status == "OK":
        final_verdict = "GOOD"
    elif status == "DEFECTIVE":
        final_verdict = "DEFECTIVE"
    else:
        final_verdict = status

    results = {
        "original_b64": original_b64,
        "yolo_init_b64": y_init_b64,
        "yolo_re_b64": y_re_b64,
        "final_verdict": final_verdict,
        "parts": parts_payload,
        "raw_json": json.dumps(res, indent=2, default=str)
    }

    return render_template_string(HTML, results=results)

flask_app = app

print("Flask app defined. Ready to start in the next cell.")


Flask app defined. Ready to start in the next cell.


In [ ]:
import requests

requests.get("http://127.0.0.1:5000/shutdown")


INFO:werkzeug:127.0.0.1 - - [07/Dec/2025 17:58:36] "GET /shutdown HTTP/1.1" 405 -


<Response [405]>

In [ ]:
import os

os.system("kill -9 $(lsof -t -i:5000)")


In [ ]:
# ------------------ Cell 2: Start Flask server in background ------------------
import threading, time

def run_flask():
    flask_app.run(host="0.0.0.0", port=5000, debug=False, use_reloader=False)

# don’t start twice
if "flask_thread" in globals() and flask_thread.is_alive():
    print("Flask is already running on port 5000.")
else:
    flask_thread = threading.Thread(target=run_flask, daemon=True)
    flask_thread.start()
    time.sleep(1.5)
    print("Flask started on port 5000.")
# ------------------ End Cell 2 ------------------


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


Flask started on port 5000.


In [ ]:
# ------------------ Cell 3: install node/npm if needed, install localtunnel, start tunnel ------------------
# Pick a short custom subdomain (letters/numbers/hyphen). Change it below.
import random # Import the random module
SUBDOMAIN = "pcbinspect" + str(random.randint(100,999))  # change to e.g. "pcbayush" if you prefer

# 1) ensure node & npm exist (install minimal Node if missing)
import shutil, subprocess, time, sys, os, re
if shutil.which("node") is None or shutil.which("npm") is None:
    print("Installing nodejs & npm (apt-get)... this may take ~20-30s")
    !apt-get update -qq
    !apt-get install -y -qq nodejs npm
else:
    print("node & npm already present.")

# 2) install localtunnel globally (if not available)
if shutil.which("lt") is None:
    print("Installing localtunnel (npm)...")
    !npm install -g localtunnel
else:
    print("localtunnel (lt) already installed.")

# 3) Try to start localtunnel using subprocess and custom subdomain.
# We'll try a few subdomain attempts if the first is taken.
max_attempts = 4
attempt = 0
lt_proc = None
public_url = None

while attempt < max_attempts and public_url is None:
    attempt += 1
    sub = SUBDOMAIN if attempt == 1 else f"{SUBDOMAIN}{attempt}"
    print(f"Attempt {attempt}: trying subdomain = {sub}")
    # use npx lt if 'lt' binary not found; otherwise use lt
    cmd = ["npx", "localtunnel", "--port", "5000", "--subdomain", sub] if shutil.which("lt") is None else ["lt", "--port", "5000", "--subdomain", sub]
    # run in background and capture stdout
    try:
        lt_proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        print("Failed to start localtunnel process:", e)
        break

    # wait up to 8 seconds for output
    start = time.time()
    while time.time() - start < 8:
        line = lt_proc.stdout.readline()
        if not line:
            time.sleep(0.1)
            continue
        print("lt:", line.strip())
        # localtunnel prints a line like: your url is: https://<subdomain>.loca.lt
        m = re.search(r"https?://[^\s]+", line)
        if m:
            public_url = m.group(0).strip()
            break

    if public_url:
        print("✔ LocalTunnel public URL:", public_url)
        print("Open it in your browser. The app endpoint is:", public_url)
        break
    else:
        # if process is still running and no url, kill and retry with another subdomain
        try:
            lt_proc.kill()
        except:
            pass
        lt_proc = None
        print("No URL from localtunnel for subdomain", sub, "- trying another subdomain...")

if public_url is None:
    print("Failed to create localtunnel after attempts. Inspect process logs or try a different SUBDOMAIN manually.")
else:
    # keep lt_proc reference so tunnel stays up; save it to globals for later control
    GLOBAL_LT_PROC = lt_proc
    GLOBAL_LT_URL = public_url
    print("Tunnel running in background. To stop it, run: GLOBAL_LT_PROC.kill() or restart the Colab runtime.")
# ------------------ End Cell 3 ------------------


node & npm already present.
localtunnel (lt) already installed.
Attempt 1: trying subdomain = pcbinspect941
lt: your url is: https://pcbinspect941.loca.lt
✔ LocalTunnel public URL: https://pcbinspect941.loca.lt
Open it in your browser. The app endpoint is: https://pcbinspect941.loca.lt
Tunnel running in background. To stop it, run: GLOBAL_LT_PROC.kill() or restart the Colab runtime.


In [ ]:
!curl -s https://loca.lt/mytunnelpassword

34.26.15.15